# Allen--Cahn: EnSF Filtering Experiments

Runs the Ensemble Score Filter (EnSF) on the Allen--Cahn equation
for 100% and 70% observation densities (Case 1: constant mobility).
These are the results reported in Chapter 3 of the thesis.

**Requires:** `../data/ac_dataset_case1.npz` from `00_generate_truth.ipynb`

**Filter:** EnSF via reverse-time SDE (Bao et al., 2024).  
The `reverse_SDE` implementation is shared with the heat equation — see `../heat_equation/ensf.py`.

**Observation model:** $Y_t = \arctan(X_t) + \varepsilon_t$

**Outputs** saved to `../data/`:
- `ac_ensf_obs100.npz`
- `ac_ensf_obs70.npz`

In [ ]:
import sys
import numpy as np
import time
from pathlib import Path
from mpi4py import MPI
from dolfinx import fem
from tqdm import tqdm

# Add heat_equation folder to path to reuse ensf.py
sys.path.append(str(Path("../heat_equation").resolve()))
from ensf import reverse_SDE, make_score_likelihood_arctan

from solver import AllenCahnIMEX, create_mesh

## 1. Load dataset

In [ ]:
DATA_DIR = Path("../data")

data         = np.load(DATA_DIR / "ac_dataset_case1.npz", allow_pickle=True)
X_true_obs   = data["X_true_obs"]       # (Nobs, state_dim)
Y            = data["Y"]                # (Nobs, state_dim)
t_obs        = data["t_obs"]            # (Nobs,)
energies_ref = data["energies"]         # reference energy trajectory
supnorm_ref  = data["supnorm"]          # reference sup norm

dt           = float(data["dt"])
alpha        = float(data["alpha"])
Nx           = int(data["Nx"])
Ny           = int(data["Ny"])
OBS_SIGMA    = float(data["obs_sigma"])
mobility_case = int(data["mobility_case"])

Nobs, state_dim = X_true_obs.shape
N = Nobs - 1
T = float(t_obs[-1])
dt_filter = T / N

print(f"State dim: {state_dim}, Nobs: {Nobs}, T: {T:.1f}, dt_filter: {dt_filter:.4f}")

## 2. Mesh and FEniCSx setup

In [ ]:
domain, V = create_mesh(Nx=Nx, Ny=Ny)
phi_est   = fem.Function(V, name="phi_est")

## 3. EnSF hyperparameters

In [ ]:
ENSEMBLE_SIZE  = 50
EPS_ALPHA      = 0.05
EULER_STEPS    = 250
FORECAST_SIGMA = 0.01   # additive noise on state particles after forecast step
XI_SIGMA       = 0.05   # stochastic mobility noise inside the solver

## 4. Filtering loop

In [ ]:
def run_experiment(obs_fraction: float, save_tag: str):
    """
    Run one EnSF experiment for a given observation fraction.

    Parameters
    ----------
    obs_fraction : float
        Fraction of state DOFs observed, in (0, 1].
    save_tag : str
        Base filename for the NPZ output.
    """
    rng = np.random.default_rng(42)

    # Observation mask
    if obs_fraction >= 1.0:
        observed_mask = np.ones(state_dim, dtype=bool)
    else:
        n_obs = int(obs_fraction * state_dim)
        obs_ids = rng.choice(state_dim, n_obs, replace=False)
        observed_mask = np.zeros(state_dim, dtype=bool)
        observed_mask[obs_ids] = True

    # Initial ensemble: random U(-0.9, 0.9), matching IC distribution
    state_particles = rng.uniform(-0.9, 0.9, size=(ENSEMBLE_SIZE, state_dim))

    # Storage
    EnSF_est_save = np.zeros((N + 1, state_dim))
    step_rmse     = np.zeros(N)
    energy_arr    = np.zeros(N)
    supnorm_arr   = np.zeros(N)
    timing_forecast = np.zeros(N)
    timing_analysis = np.zeros(N)

    EnSF_est_save[0] = state_particles.mean(axis=0)

    # Reuse one solver instance across the filtering loop
    problem = AllenCahnIMEX(
        V,
        dt=dt_filter,
        alpha=alpha,
        mobility_case=mobility_case,
        xi_sigma=XI_SIGMA,
        noise_is_spatial=True,
    )

    for i in tqdm(range(1, N + 1), desc=f"{save_tag} (obs={obs_fraction:.0%})"):

        # ---- Forecast step ----
        t0 = time.perf_counter()
        for j in range(ENSEMBLE_SIZE):
            problem.set_initial_condition(state_particles[j])
            phi_next, _, _ = problem.step()
            # additive model noise on top of solver step
            state_particles[j] = (
                phi_next.x.array[:state_dim]
                + np.sqrt(dt_filter) * FORECAST_SIGMA * rng.standard_normal(state_dim)
            )
        timing_forecast[i - 1] = time.perf_counter() - t0

        # ---- Analysis step ----
        obs = Y[i].copy()
        score_fn = make_score_likelihood_arctan(obs, observed_mask, OBS_SIGMA)

        t0 = time.perf_counter()
        state_particles = reverse_SDE(
            x0=state_particles,
            ensemble_size=ENSEMBLE_SIZE,
            state_dim=state_dim,
            eps_alpha=EPS_ALPHA,
            score_likelihood=score_fn,
            time_steps=EULER_STEPS,
        )
        timing_analysis[i - 1] = time.perf_counter() - t0

        # Clip to valid phase-field range
        np.clip(state_particles, -1.0 + 1e-5, 1.0 - 1e-5, out=state_particles)

        # ---- Diagnostics ----
        EnSF_est = state_particles.mean(axis=0)
        EnSF_est_save[i] = EnSF_est

        phi_est.x.array[:state_dim] = EnSF_est
        phi_est.x.scatter_forward()

        step_rmse[i - 1]   = np.sqrt(np.mean((X_true_obs[i] - EnSF_est) ** 2))
        energy_arr[i - 1]  = problem.compute_energy(phi_est)
        supnorm_arr[i - 1] = float(np.max(np.abs(EnSF_est)))

        if i % 50 == 0:
            print(f"  step {i:3d}/{N} | RMSE={step_rmse[i-1]:.4f} "
                  f"| supnorm={supnorm_arr[i-1]:.3f} "
                  f"| E={energy_arr[i-1]:.4f}")

    # ---- Save ----
    out_path = DATA_DIR / f"{save_tag}.npz"
    np.savez(
        out_path,
        EnSF_est      = EnSF_est_save,
        step_rmse     = step_rmse,
        energy_arr    = energy_arr,
        supnorm_arr   = supnorm_arr,
        t_obs         = t_obs,
        observed_mask = observed_mask.astype(np.uint8),
        timing_forecast = timing_forecast,
        timing_analysis = timing_analysis,
        # metadata
        obs_fraction   = obs_fraction,
        ensemble_size  = ENSEMBLE_SIZE,
        euler_steps    = EULER_STEPS,
        eps_alpha      = EPS_ALPHA,
        forecast_sigma = FORECAST_SIGMA,
        xi_sigma       = XI_SIGMA,
    )
    print(f"Saved → {out_path}")

## 5. Run experiments

In [ ]:
run_experiment(obs_fraction=1.0, save_tag="ac_ensf_obs100")
run_experiment(obs_fraction=0.7, save_tag="ac_ensf_obs70")